 Z-Image
Generate, edit and upscale with Z-Image, at http://zimage:7860 on Tailscale.

1. **Runtime > Change runtime type > T4 GPU**
2. **Secrets** (key icon, left), allow notebook access: `HF_TOKEN` (gated or private repos),
   `CIVITAI_TOKEN` (Civitai downloads), `TS_AUTHKEY` (Tailscale, tick Reusable + Ephemeral; only the app notebooks).
3. Run **Setup** (re-runs skip finished steps), then the cells below in order.

In [ ]:
#@title Setup: repo, venv, secrets, Drive
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "step5-notebooks"  #@param {type:"string"}
USE_DRIVE = False  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/SDXLR"  #@param {type:"string"}
GROUPS = 'app'  # venv install groups for this notebook (None = no shared venv)
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # follow BRANCH even if an earlier run cloned another one
    subprocess.run(["git", "fetch", "-q", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# The kernel only orchestrates (stdlib); GPU work runs in venv subprocesses,
# so this kernel never holds VRAM.
sys.path.insert(0, str(REPO / "src"))
from sdxlr import nb
if GROUPS is not None:
    r = subprocess.run([sys.executable, "-m", "sdxlr.bootstrap", "--groups", GROUPS], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True, env={**os.environ, "PYTHONPATH": str(REPO / "src")})
    print(r.stdout[-3000:])
    assert r.returncode == 0, "setup failed (see above)"
found = nb.load_secrets()
print("secrets:", ", ".join(f"{k} {'found' if v else 'missing'}" for k, v in found.items()))
ROOT = nb.drive_mount(DRIVE_ROOT) if USE_DRIVE else None
print("drive:", ROOT or "off")

# 1. Models (optional)
Downloads now so the app loads fast; you can also pick everything inside the app.
A custom Z-Image checkpoint (HF file, Civitai or https link, or Drive path) is loaded as 4-bit the first time
and cached. Models in `SDXLR/models/zimage` and LoRAs in `SDXLR/loras/zimage` are listed in the app when Drive is on.

In [ ]:
#@title Models
MODEL = "turbo 4-bit (unsloth)"  #@param ["turbo 4-bit (unsloth)", "turbo Q8", "turbo Q6", "turbo Q5", "turbo Q4"]
CUSTOM_MODEL = ""  #@param {type:"string"}
TE = "abliterated"  #@param ["abliterated", "official", "heretic", "engineer"]
#@markdown TE: use the one your LoRA was trained with. `abliterated`/`heretic` follow prompts without refusing;
#@markdown `engineer` also writes the best prompts with **Write a detailed prompt**.
LORAS = ""  #@param {type:"string"}
#@markdown LORAS: comma list of LoRA links or .zip files.
from sdxlr import paths
if CUSTOM_MODEL.strip():
    nb.run(["-m", "sdxlr.fetch", "zimage", CUSTOM_MODEL.strip()])
else:
    nb.run(["-m", "sdxlr.fetch", "prefetch-zimage", paths.Z_MODELS[MODEL], paths.Z_ENCODERS[TE]])
srcs = [x.strip() for x in LORAS.split(",") if x.strip()]
if srcs:
    nb.run(["-m", "sdxlr.fetch", "loras", *srcs])

# 2. Launch
Open **http://zimage:7860** on a device signed in to your Tailscale (needs TS_AUTHKEY in Secrets).
Pages: Generate (with edit and upscale), Library, History, Activity. Pick the same TE in the app as above.
**Keep this Colab tab open** and click in it now and then: Colab can stop an idle notebook (often after about
90 minutes) even while you use the app. The cell keeps running while the app is open; stop it to close the app.

In [ ]:
#@title Launch the app
args = ["-m", "sdxlr.webui", "--port", 7860, "--host-name", "zimage", "--zimage-only"]
if ROOT:
    args += ["--drive", ROOT]
nb.serve(args)  # starts the app again if it crashes

# Finish
Saves Drive, then **Runtime > Disconnect and delete runtime** (pictures on the VM are gone with it).

In [ ]:
#@title Finish
if ROOT:
    from sdxlr import drive
    drive.flush_and_unmount()